# T1-13 Agentic boundary + ticket routing (Track A)

**Time:** 13:30–14:15 &nbsp;|&nbsp; **Evidence:** ticket scoreboard + allow/ban list

23 training rows is **very small**. A weak result is a finding, not a failure of the lab.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** change only lines marked `# [CHANGE]`. Allowed values are named in the cell above.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path.

Git is not required. If you are stuck, load `completed/` for **this** notebook only and mark it as loaded from reference.


**Data ban:** CivicDesk fiction only. `heldout/cases.py` is a **test** set — do not train on it.


## 0. Backend


In [ ]:
# [CHANGE] olmo = live local model. demo = RECORDED / labelled cards only (not this session).
BACKEND = "olmo"  # "olmo" | "phi" | "demo"

import civicdesk_ft as ft

cfg = ft.set_backend(BACKEND)


## 1. Base on the 9 held-out tickets, live

What this cell does: pre-flight tickets (or reuse today's file), then show T1 and T8 live — not only the card.


In [ ]:
# [RUN]
import civicdesk_ft as ft

record = ft.run_preflight(["tickets"])
ft.show_failure("T1")
print()
# T8 is injection — show from live file if present
from pathlib import Path
import json

pf = Path("logs/preflight_latest.json")
if pf.is_file():
    blob = json.loads(pf.read_text(encoding="utf-8"))
    raw = (blob.get("outputs") or {}).get("tickets", {}).get("T8")
    if raw and blob.get("measured"):
        print("LIVE, this session — T8")
        print(raw)
    else:
        print("T8 not in this session's measured pre-flight; open heldout/cases.py for the gold.")


## 2. Ticket training mix


In [ ]:
# [RUN]
import runpy
from pathlib import Path
import civicdesk_ft as ft

runpy.run_path(str(Path("build_tickets_set.py")), run_name="__main__")
ft.inspect_ticket_rows()


## 3. Train a ticket adapter (small data on purpose)


In [ ]:
# [RUN]
import ft_train

TARGETS = ["q_proj", "v_proj"]
RANK = 8
try:
    session = ft_train.TrainSession(task="tickets", targets=TARGETS, rank=RANK)
    session.summary()
    session.show_example(0)
except RuntimeError as exc:
    session = None
    print("SKIP train:", exc)


In [ ]:
# [RUN] one update; rerun to continue
if session is not None:
    session.step(1)


In [ ]:
# [CHANGE] more updates (e.g. 15). Small data — do not expect miracles.
MORE = 15
if session is not None:
    session.step(MORE)
    session.history()
    session.validate()
    session.save("adapters/my_tickets_run")


## 4. Score all 9 held-out tickets


In [ ]:
# [CHANGE]
ADAPTER = "adapters/my_tickets_run"  # or "prebuilt"

import ft_train

try:
    model = ft_train.load_adapter(ADAPTER, task="tickets")
    compare = ft_train.compare_live(model, "tickets")
    ft_train.ticket_scoreboard(compare)
except Exception as exc:
    print("Live tickets compare unavailable:", exc)
    try:
        compare = ft_train.load_recorded("tickets")
        ft_train.ticket_scoreboard(compare)
    except Exception as exc2:
        print(exc2)


## 5. Tool gate (rule code, not model behaviour)

The gate is ordinary Python. The model may *propose* an action; the gate decides.


In [ ]:
# [RUN]
import civicdesk_ft as ft

for action in [
    "draft_ticket",
    "quote_handbook",
    "draft_redacted_copy",
    "reset_password",
    "write_glpi",
    "award_contract",
    "scrape_web_for_negotiation",
    "release_to_panel",
    "auto_release_if_pass_checks",
]:
    print(f"{action:32} {ft.tool_gate(action)}")


In [ ]:
# [RUN] model proposes; gate decides
PROPOSED = "reset_password"  # pretend the ticket adapter suggested this
import civicdesk_ft as ft

print("model proposed:", PROPOSED)
print("tool_gate:", ft.tool_gate(PROPOSED))
print("The boundary is the gate, not the LoRA.")


In [ ]:
# [CHANGE] try grant_ad_rights, publish_notice, deploy_patrol
ACTION = "grant_ad_rights"
import civicdesk_ft as ft

print(ACTION, ft.tool_gate(ACTION))
